# Notebook 01 - Consolidacion de datos fiscales

**Fuente:** Secretaria de Hacienda - https://www.argentina.gob.ar/economia/sechacienda/infoestadistica  
**Datos:** Sector Publico Base Caja (AIF) + Informe Mensual de Ingresos y Gastos (IMIG)  

Este notebook lee los datasets consolidados desde GitHub y los exporta como un unico archivo Excel con multiples hojas, listo para usar en el **Notebook 02 de analisis**.

> **Como agregar nuevos meses:** ver instrucciones en el README del repositorio.

In [ ]:
# Celda 1 - Instalacion de dependencias
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q pandas openpyxl

import pandas as pd
import openpyxl
import os
print('OK - dependencias cargadas')

In [ ]:
# Celda 2 - URLs de los datos (rama main)
REPO_RAW = 'https://raw.githubusercontent.com/santiagoriverti/cuentas_publicas/main'

AIF_URL  = f'{REPO_RAW}/output/aif_consolidado.csv'
IMIG_URL = f'{REPO_RAW}/output/imig_consolidado.csv'

print('Descargando datasets...')
df_aif  = pd.read_csv(AIF_URL,  parse_dates=['fecha'])
df_imig = pd.read_csv(IMIG_URL, parse_dates=['fecha'])

print(f'AIF : {len(df_aif):,} registros | {df_aif["fecha"].min().strftime("%Y-%m")} a {df_aif["fecha"].max().strftime("%Y-%m")}')
print(f'IMIG: {len(df_imig):,} registros')
df_aif.head(3)

In [ ]:
# Celda 3 - Resumen de cobertura
print('=== AIF - Conceptos disponibles ===')  
print(df_aif['concepto_codigo'].value_counts().to_string())
print()
print('=== AIF - Subsectores ===')
print(df_aif['subsector'].value_counts().to_string())
print()
print('=== Meses cubiertos por anio ===')
print(df_aif[df_aif['periodo']=='mensual'].groupby('anio')['mes'].nunique().to_string())

In [ ]:
# Celda 4 - Exportar a Excel consolidado
OUTPUT_FILE = 'datos_fiscales_consolidado.xlsx'

df_mensual = df_aif[df_aif['periodo'] == 'mensual']

# ── Hoja: Resultado_pivot ─────────────────────────────────────────────
# Una fila por mes, una columna por concepto clave (total Adm. Nacional)
df_resultados = df_mensual[
    (df_mensual['subsector'] == 'total_adm_nacional') &
    (df_mensual['concepto_codigo'].isin([
        'I_INGRESOS_CORRIENTES',
        'II_GASTOS_CORRIENTES',
        'II2_INTERESES',
        'II3_PRESTACIONES_SEG_SOCIAL',
        'XIV_RESULTADO_PRIMARIO',
        'XV_RESULTADO_FINANCIERO',
    ]))
].pivot_table(
    index='fecha', columns='concepto_codigo', values='valor_millones_pesos'
).reset_index()

# ── Hoja: Transferencias_provincias ──────────────────────────────────
# Transferencias corrientes Y de capital a Provincias/CABA
# Incluye todos los subsectores para poder sumar el total
df_prov = df_mensual[
    df_mensual['concepto_codigo'].isin([
        'II4b1_TRANSF_PROVINCIAS_CABA',    # corrientes
        'V2a_TRANSF_CAPITAL_PROVINCIAS',   # capital
    ])
].copy()

# Pivot: una fila por mes/tipo, columnas = subsectores
df_prov_pivot = df_prov.pivot_table(
    index=['fecha', 'concepto_codigo'],
    columns='subsector',
    values='valor_millones_pesos'
).reset_index()
df_prov_pivot.columns.name = None

# Renombrar concepto para legibilidad
df_prov_pivot['tipo'] = df_prov_pivot['concepto_codigo'].map({
    'II4b1_TRANSF_PROVINCIAS_CABA':  'Corrientes',
    'V2a_TRANSF_CAPITAL_PROVINCIAS': 'Capital',
})
cols_order = ['fecha', 'tipo'] + [c for c in df_prov_pivot.columns if c not in ('fecha','tipo','concepto_codigo')]
df_prov_pivot = df_prov_pivot[cols_order]

# ── Exportar ──────────────────────────────────────────────────────────
print(f'Exportando a {OUTPUT_FILE}...')
with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:

    df_mensual.to_excel(
        writer, sheet_name='AIF_mensual', index=False)

    df_aif[df_aif['periodo'] == 'acumulado'].to_excel(
        writer, sheet_name='AIF_acumulado', index=False)

    df_resultados.to_excel(
        writer, sheet_name='Resultado_pivot', index=False)

    df_prov_pivot.to_excel(
        writer, sheet_name='Transferencias_provincias', index=False)

    df_imig.to_excel(
        writer, sheet_name='IMIG', index=False)

print(f'Archivo generado: {OUTPUT_FILE}')
print()
print('Hojas exportadas:')
print('  AIF_mensual              - todos los conceptos x subsector, periodicidad mensual')
print('  AIF_acumulado            - idem, acumulado anual')
print('  Resultado_pivot          - KPIs clave en columnas: ingresos, gastos, intereses, resultado')
print('  Transferencias_provincias- corrientes Y capital a provincias/CABA, por subsector')
print('  IMIG                     - detalle funcional de ingresos y gastos')
print()
print(f'Cobertura: {df_aif["fecha"].min().strftime("%Y-%m")} a {df_aif["fecha"].max().strftime("%Y-%m")}')
print(f'Meses cubiertos: {df_mensual["fecha"].nunique()}')

In [ ]:
# Celda 5 - Verificacion rapida de transferencias a provincias
print('Transferencias Tesoro -> Provincias/CABA (ultimos 12 meses, MM ARS):')
print()
for tipo, cod in [('Corrientes', 'II4b1_TRANSF_PROVINCIAS_CABA'), ('Capital', 'V2a_TRANSF_CAPITAL_PROVINCIAS')]:
    serie = df_mensual[
        (df_mensual['concepto_codigo'] == cod) &
        (df_mensual['subsector'] == 'tesoro_nacional')
    ].set_index('fecha')['valor_millones_pesos'].sort_index()
    print(f'  {tipo}:')
    print(serie.tail(12).to_string())
    print()

In [ ]:
# Celda 6 - Descargar el Excel (solo en Colab)
if IN_COLAB:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print('Descarga iniciada')
else:
    print(f'Archivo guardado en: {os.path.abspath(OUTPUT_FILE)}')